## 1. Setup and Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")

# Input: Merged StockTwits-CRSP data by year
INPUT_FOLDER = DATA_DIR / "merged_with_crsp_mlcrowd"

# Output: Features (consolidated pickle files)
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_FOLDER / "features_01_basic_sentiment.pkl"

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Output file: {OUTPUT_FILE}")

## 2. Load Sample Data for Development & Testing

In [ ]:
# Get list of available files
files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.csv')])
print(f"Available files: {len(files)}")
print(f"Files: {files}")

# Load most recent year for development
if files:
    sample_file = INPUT_FOLDER / files[-1]
    print(f"\nLoading sample file for testing: {sample_file.name}")
    
    df_sample = pd.read_csv(sample_file)
    print(f"\nShape: {df_sample.shape}")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nSample data:")
    display(df_sample.head())
    
    # Check date range
    df_sample['date'] = pd.to_datetime(df_sample['date'])
    print(f"\nDate range: {df_sample['date'].min()} to {df_sample['date'].max()}")
    print(f"Unique dates: {df_sample['date'].nunique():,}")
    print(f"Unique symbols: {df_sample['symbol'].nunique():,}")
    print(f"Unique users: {df_sample['user_id'].nunique():,}")

## 3. Define Feature Calculation Function

In [ ]:
def calculate_basic_sentiment_features(df):
    """
    Calculate basic sentiment ratios and consensus metrics for each stock-day.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Merged StockTwits-CRSP data with columns: symbol, date, sentiment
        
    Returns:
    --------
    pd.DataFrame: Stock-day level features including:
        - bullish_ratio, bearish_ratio, net_sentiment
        - extreme_bullish_80, extreme_bullish_90
        - extreme_bearish_80, extreme_bearish_90
        - disagreement_index
    """
    # Ensure date is datetime
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    # Group by symbol and date
    grouped = df.groupby(['symbol', 'date'])
    
    # Count bullish and bearish messages
    sentiment_counts = grouped['sentiment'].value_counts().unstack(fill_value=0)
    
    # Ensure both columns exist (some days might have only bullish or bearish)
    if 'Bullish' not in sentiment_counts.columns:
        sentiment_counts['Bullish'] = 0
    if 'Bearish' not in sentiment_counts.columns:
        sentiment_counts['Bearish'] = 0
    
    # Calculate total labeled messages (excluding neutral/unlabeled)
    sentiment_counts['total_labeled'] = sentiment_counts['Bullish'] + sentiment_counts['Bearish']
    
    # Calculate ratios (handle division by zero)
    sentiment_counts['bullish_ratio'] = np.where(
        sentiment_counts['total_labeled'] > 0,
        sentiment_counts['Bullish'] / sentiment_counts['total_labeled'],
        np.nan
    )
    
    sentiment_counts['bearish_ratio'] = np.where(
        sentiment_counts['total_labeled'] > 0,
        sentiment_counts['Bearish'] / sentiment_counts['total_labeled'],
        np.nan
    )
    
    sentiment_counts['net_sentiment'] = np.where(
        sentiment_counts['total_labeled'] > 0,
        (sentiment_counts['Bullish'] - sentiment_counts['Bearish']) / sentiment_counts['total_labeled'],
        np.nan
    )
    
    # --- New Features (Dec 24, 2025) ---
    
    # Features 12 & 13: Extreme Bullish Consensus
    sentiment_counts['extreme_bullish_80'] = (sentiment_counts['bullish_ratio'] > 0.8).astype(int)
    sentiment_counts['extreme_bullish_90'] = (sentiment_counts['bullish_ratio'] > 0.9).astype(int)
    
    # Features 13b & 13c: Extreme Bearish Consensus
    sentiment_counts['extreme_bearish_80'] = (sentiment_counts['bearish_ratio'] > 0.8).astype(int)
    sentiment_counts['extreme_bearish_90'] = (sentiment_counts['bearish_ratio'] > 0.9).astype(int)
    
    # Feature 39: Disagreement Index
    # Formula: 1 - |N_Bull - N_Bear| / (N_Bull + N_Bear)
    # This is equivalent to 1 - abs(net_sentiment)
    sentiment_counts['disagreement_index'] = 1 - sentiment_counts['net_sentiment'].abs()
    
    # Select final columns
    features = sentiment_counts[[
        'Bullish', 'Bearish', 'total_labeled', 
        'bullish_ratio', 'bearish_ratio', 'net_sentiment',
        'extreme_bullish_80', 'extreme_bullish_90',
        'extreme_bearish_80', 'extreme_bearish_90',
        'disagreement_index'
    ]].copy()
    
    # Rename count columns for clarity
    features = features.rename(columns={
        'Bullish': 'n_bullish',
        'Bearish': 'n_bearish'
    })
    
    return features.reset_index()

## 4. Test on Sample Data

In [ ]:
# Calculate features for sample data
features_sample = calculate_basic_sentiment_features(df_sample)

print(f"Features shape: {features_sample.shape}")
print(f"\nFeature columns: {list(features_sample.columns)}")
print(f"\nSample features:")
display(features_sample.head(20))

# Summary statistics
print(f"\nSummary statistics:")
display(features_sample[['n_bullish', 'n_bearish', 'total_labeled', 
                          'bullish_ratio', 'bearish_ratio', 'net_sentiment']].describe())

## 5. Visualize & Validate Features

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Filter out NaN values for visualization
features_vis = features_sample.dropna(subset=['bullish_ratio', 'bearish_ratio', 'net_sentiment'])

# 1. Bullish Ratio distribution
axes[0, 0].hist(features_vis['bullish_ratio'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Bullish Ratio')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Bullish Ratio')
axes[0, 0].axvline(features_vis['bullish_ratio'].median(), color='red', 
                    linestyle='--', label=f"Median: {features_vis['bullish_ratio'].median():.3f}")
axes[0, 0].legend()

# 2. Bearish Ratio distribution
axes[0, 1].hist(features_vis['bearish_ratio'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Bearish Ratio')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Bearish Ratio')
axes[0, 1].axvline(features_vis['bearish_ratio'].median(), color='red', 
                    linestyle='--', label=f"Median: {features_vis['bearish_ratio'].median():.3f}")
axes[0, 1].legend()

# 3. Net Sentiment distribution
axes[1, 0].hist(features_vis['net_sentiment'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Net Sentiment')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Net Sentiment')
axes[1, 0].axvline(features_vis['net_sentiment'].median(), color='red', 
                    linestyle='--', label=f"Median: {features_vis['net_sentiment'].median():.3f}")
axes[1, 0].axvline(0, color='black', linestyle='-', alpha=0.3)
axes[1, 0].legend()

# 4. Bullish vs Bearish scatter
axes[1, 1].scatter(features_vis['bullish_ratio'], features_vis['bearish_ratio'], 
                   alpha=0.3, s=10)
axes[1, 1].set_xlabel('Bullish Ratio')
axes[1, 1].set_ylabel('Bearish Ratio')
axes[1, 1].set_title('Bullish vs Bearish Ratio')
axes[1, 1].plot([0, 1], [1, 0], 'r--', alpha=0.5, label='Sum = 1')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Print validation checks
print(f"\nValidation Checks:")
print(f"="*60)
print(f"Bullish + Bearish ratio should ≈ 1.0:")
ratio_sum = features_vis['bullish_ratio'] + features_vis['bearish_ratio']
print(f"  Mean sum: {ratio_sum.mean():.6f}")
print(f"  Min sum: {ratio_sum.min():.6f}")
print(f"  Max sum: {ratio_sum.max():.6f}")

print(f"\nNet sentiment should be in [-1, 1]:")
print(f"  Min: {features_vis['net_sentiment'].min():.6f}")
print(f"  Max: {features_vis['net_sentiment'].max():.6f}")

print(f"\nOverall sentiment bias:")
print(f"  Mean bullish ratio: {features_vis['bullish_ratio'].mean():.3f}")
print(f"  Mean bearish ratio: {features_vis['bearish_ratio'].mean():.3f}")
print(f"  Mean net sentiment: {features_vis['net_sentiment'].mean():.3f}")

## 6. Process All Years

In [ ]:
# Process all years and concatenate
all_features = []
processing_stats = []

print(f"{'='*60}")
print(f"Processing all years...")
print(f"{'='*60}\n")

for file in tqdm(files, desc="Processing years"):
    try:
        # Load year data
        year = file.split('_')[-1].replace('.csv', '')
        file_path = INPUT_FOLDER / file
        
        df_year = pd.read_csv(file_path)
        
        # Calculate features
        features_year = calculate_basic_sentiment_features(df_year)
        
        # Add to list
        all_features.append(features_year)
        
        # Track stats
        processing_stats.append({
            'year': year,
            'input_rows': len(df_year),
            'feature_rows': len(features_year),
            'unique_symbols': features_year['symbol'].nunique(),
            'unique_dates': features_year['date'].nunique()
        })
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")

# Concatenate all years
features_all = pd.concat(all_features, ignore_index=True)

print(f"\n{'='*60}")
print(f"Processing Complete!")
print(f"{'='*60}")
print(f"\nTotal features: {len(features_all):,}")
print(f"Unique symbols: {features_all['symbol'].nunique():,}")
print(f"Date range: {features_all['date'].min()} to {features_all['date'].max()}")

# Show processing stats
df_stats = pd.DataFrame(processing_stats)
print(f"\nProcessing Statistics by Year:")
display(df_stats)

## 7. Final Data Inspection

In [ ]:
# Inspect final dataset
print(f"Final Features Dataset:")
print(f"="*60)
print(f"Shape: {features_all.shape}")
print(f"\nColumns: {list(features_all.columns)}")
print(f"\nData types:")
print(features_all.dtypes)
print(f"\nMemory usage: {features_all.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nNull values:")
print(features_all.isnull().sum())
print(f"\nSample data:")
display(features_all.head(20))
print(f"\nSummary statistics:")
display(features_all[['n_bullish', 'n_bearish', 'total_labeled', 
                       'bullish_ratio', 'bearish_ratio', 'net_sentiment']].describe())

## 8. Save to Pickle

In [ ]:
# Save features to pickle
print(f"Saving features to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)
print(f"✓ Saved successfully!")

# Verify save
print(f"\nVerifying saved file...")
features_loaded = pd.read_pickle(OUTPUT_FILE)
print(f"✓ File readable")
print(f"✓ Shape matches: {features_loaded.shape == features_all.shape}")
print(f"✓ Columns match: {list(features_loaded.columns) == list(features_all.columns)}")

# File size
file_size_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"\nFile size: {file_size_mb:.2f} MB")

## Summary

✓ **Features Calculated:**
- **Basic Sentiment:**
    - Bullish Ratio
    - Bearish Ratio  
    - Net Sentiment
- **Extreme Consensus:**
    - Extreme Bullish Consensus (80% / 90%)
    - Extreme Bearish Consensus (80% / 90%)
- **Network Structure:**
    - Disagreement Index

✓ **Output:** `features_01_basic_sentiment.pkl`

✓ **Ready for:** Feature aggregation in final pipeline